# FunctionGraph & Function LaText to parquet
This notebook shows the final version of the cells used to generate the synthetic dataset: [Croc-Prog-HF/Simplified_FunctionGraph-LaTeX](https://huggingface.co/datasets/Croc-Prog-HF/Simplified_FunctionGraph-LaTeX)<br/>

### Subset: Graph-function_static
This code is used to generate functions and graphs that are consistent and very similar to each other in terms of graphics. It's useful for the model's initial understanding of the relationship between functions and graphs.

In [ ]:
!pip install pandas pyarrow
!pip install datasets
!pip install antlr4-python3-runtime==4.11

In [ ]:
import random
import io
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy import symbols, sqrt, log, sin, cos, tan, exp, init_printing, latex, lambdify
from datasets import Dataset, Features, Value, Image

# ==============
# Formula latex
# ==============
def GenFx():
    init_printing(use_latex=True)
    x = symbols('x')
    
    def genera_monomio():
        scelta_base = random.randint(1, 4)
        if scelta_base == 1: # Caso Radice
            op = random.choice([x, log(x), sin(x), cos(x), tan(x), sin(x)/cos(x)])
            return sqrt(op)
        elif scelta_base == 2: # Caso Logaritmo
            base = random.choice([2, 3, 5, 10, 'e'])
            arg_op = random.choice([sin(x), cos(x), tan(x)])
            return log(arg_op) if base == 'e' else log(arg_op, base)
        elif scelta_base == 3: # Caso Goniometria
            return random.choice([sin(x), cos(x), tan(x), exp(x)])
        else: # Caso Potenza
            n = random.randint(2, 5)
            base_op = random.choice([sqrt(x), log(x, 2), 2])
            return base_op**n

    n_mnm = random.randint(2, 4)
    monomi = [genera_monomio() for _ in range(n_mnm)]
    f_custom = monomi[0]

    for i in range(1, n_mnm):
        op_scelto = random.choice(['+', '-', '*'])
        if op_scelto == '+': f_custom += monomi[i]
        elif op_scelto == '-': f_custom -= monomi[i]
        elif op_scelto == '*': f_custom *= monomi[i]
    
    return f_custom

# =============
# Generatore di grafici
# =============
def GenGraph(f_input):
    x_sym = symbols('x')
    try:
        f_expr = f_input
        f_num = lambdify(x_sym, f_expr, modules=['numpy'])

        x_vals = np.linspace(0.1, 10, 1000)
        y_vals = f_num(x_vals)

        y_vals = np.array(y_vals, dtype=np.complex128)
        y_vals = np.real(y_vals) 
        y_vals[~np.isfinite(y_vals)] = np.nan 

        if np.all(np.isnan(y_vals)):
            return None
        else:
            plt.figure(figsize=(10, 6))
            # Nessuna label passata -> nessuna legenda creata
            plt.plot(x_vals, y_vals) 
            plt.axhline(0, color='black', linewidth=0.5)
            plt.axvline(0, color='black', linewidth=0.5)
            plt.grid(True, linestyle='--', alpha=0.7)

            buf = io.BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)

            # Visualizzazione nel log di output
            plt.show() 
            plt.close()
            return buf.getvalue()
    except Exception as e:
        print(f"Error in GenGraph: {e}")
        return None

# =============
# Ciclo di generazione
# =============
TIMER = 550  # Secondi di esecuzione
start_time = time.time()
data_list = []

print("Inizio generazione dei dati...")
while time.time() - start_time < TIMER:
    f_expr = GenFx()
    latex_str = latex(f_expr)
    img_bytes = GenGraph(f_expr)

    if img_bytes is not None:
        data_list.append({
            "graph": img_bytes,
            "latex_formula": latex_str
        })

# =============
# Esportazione Locale
# =============
if data_list:
    df = pd.DataFrame(data_list)
    
    # Definiamo le features di Hugging Face (Arrow format)
    features = Features({
        "graph": Image(),
        "latex_formula": Value("string")
    })
    
    # Convertiamo in Dataset HF per gestire correttamente la colonna Image
    dataset = Dataset.from_pandas(df, features=features)
    
    # Salvataggio in formato Parquet con preservazione dei metadati Arrow
    dataset.to_parquet("Graph-function_static.parquet")
    
    print(f"\n--- Operazione completata ---")
    print(f"Righe generate: {len(dataset)}")
    print(f"File salvato correttamente come: Graph-function_static.parquet")

### Subset: Graph-function_noisy
This code is used to generate functions and graphs with changing graph images that introduce noise. It's useful for making the model robust to a wide range of graphs that aren't always well-formatted.

In [ ]:
import random
from sympy import symbols, sqrt, log, sin, cos, tan, exp, init_printing, latex
import numpy as np
import io
import time
import matplotlib.pyplot as plt
from sympy import lambdify, symbols, latex
from sympy.parsing.latex import parse_latex
import pandas as pd
from datasets import Dataset, Features, Value, Image

# ==============
# Formula latext
# ==============
def GenFx():
  init_printing(use_latex=True)
  x = symbols('x')
  y = symbols('y')

  def genera_monomio():
      scelta_base = random.randint(1, 4)

      if scelta_base == 1: # Caso Radice
          op = random.choice([
              x,
              log(x),
              sin(x),
              cos(x),
              tan(x),
              sin(x)/cos(x),
              sin(x)/tan(x),
              cos(x)/sin(x),
              cos(x)/tan(x)
          ])
          return sqrt(op)

      elif scelta_base == 2: # Caso Logaritmo
          base = random.choice([2, 3, 4, 5, 6, 7, 8, 9, 10, 'e'])
          arg_op = random.choice([sin(x), cos(x), tan(x)])
          if base == 'e':
              return log(arg_op)
          else:
              return log(arg_op, base)

      elif scelta_base == 3: # Caso Goniometria
          op = random.choice([
              sin(x),
              cos(x),
              tan(x),
              1/sin(x),
              1/cos(x),
              1/tan(x),
              exp(x),
              sin(x)/cos(x),
          ])
          return op

      elif scelta_base == 4: # Caso Potenza
          n = random.randint(2, 15)
          base_op = random.choice([
              sqrt(x),
              log(x, random.randint(2, 10)),
              random.randint(2, 500)
          ])
          return base_op**n

  # 1. Sceglie il numero di monomi
  n_mnm = random.randint(2, 6)
  # 2. Genera la funzione usando operatori casuali (+, -, *, /)
  monomi = [genera_monomio() for _ in range(n_mnm)]
  f_custom = monomi[0]

  for i in range(1, n_mnm):
      op_scelto = random.choice(['+', '-', '*', '/'])
      if op_scelto == '+':
          f_custom = f_custom + monomi[i]
      elif op_scelto == '-':
          f_custom = f_custom - monomi[i]
      elif op_scelto == '*':
          f_custom = f_custom * monomi[i]
      elif op_scelto == '/':
          f_custom = f_custom / monomi[i]
  #print(f"Numero di monomi (n_mnm): {n_mnm}")
  #display(f_custom)
  print("\nLaTeX grezzo:")
  print(latex(f_custom))
  return f_custom

# =============
# Generatore di grafici
# =============
def GenGraph(f_input):
  # Variabile simbolica
  x_sym = symbols('x')

  try:
      f_expr = f_input

      if f_expr is None:
          raise ValueError("NULL")

      # Creazione della funzione numerica
      f_num = lambdify(x_sym, f_expr, modules=['numpy'])

      # Definizione del dominio per il grafico
      x_vals = np.linspace(0.1, 10, 1000)
      # Calcolo dei valori y
      y_vals = f_num(x_vals)

      # Converti in array numpy "vero"
      y_vals = np.array(y_vals, dtype=np.complex128)
      y_vals = np.real(y_vals) # Prendi solo parte reale
      y_vals[~np.isfinite(y_vals)] = np.nan # Pulisci valori problematici

      if np.all(np.isnan(y_vals)):
          print("NULL")
          return None
      else:
          plt.figure(figsize=(10, 6))

          # --- Personalizzazioni Casuali ---
          colori = ['red', 'blue', 'black']
          stili = ['-', '--', ':', '-.']
          spessori = [1, 1.5, 2, 2.5]

          colore_scelto = random.choice(colori)
          stile_scelto = random.choice(stili)
          spessore_scelto = random.choice(spessori)
          mostra_numeri = random.choice([True, False])
          mostra_griglia = random.choice([True, False])

          # Disegno della funzione con parametri casuali
          plt.plot(x_vals, y_vals,
                   label=f"$f(x) = {latex(f_expr)}$",
                   color=colore_scelto,
                   linestyle=stile_scelto,
                   linewidth=spessore_scelto)

          # Assi cartesiani
          plt.axhline(0, color='black', linewidth=0.5)
          plt.axvline(0, color='black', linewidth=0.5)

          # Griglia casuale
          if mostra_griglia:
              plt.grid(True, linestyle='--', alpha=0.5)

          # Numeri sugli assi (Tick labels)
          if not mostra_numeri:
              plt.gca().set_xticklabels([])
              plt.gca().set_yticklabels([])

          # salvataggio
          buf = io.BytesIO()
          plt.savefig(buf, format='png')
          buf.seek(0)

          plt.show()
          plt.close()
      return buf.getvalue()
  except Exception as e:
      print(f"Error in GenGraph: {e}")
      return None


TIMER = 550  # Secondi di esecuzione
start_time = time.time()
data_list = []

while time.time() - start_time < TIMER:
    f_expr = GenFx()
    latex_str = latex(f_expr)
    img_bytes = GenGraph(f_expr)

    if img_bytes is not None:
        data_list.append({
            "graph": {
                "bytes": img_bytes,
                "path": None
            },
            "latex_formula": latex_str
        })

df = pd.DataFrame(data_list)
features = Features({
    "graph": Image(),
    "latex_formula": Value("string")
})
dataset = Dataset.from_pandas(df, features=features)
dataset.to_parquet("Graph-function_noisy.parquet")
print(f"Salvate {len(dataset)} funzioni in funzioni.parquet con metadata HF")

### Subset: Graph-function_elementary
This script allows the generation of functions that have a maximum of 3 monomials and are limited to trigonometric functions, logarithms, powers, and roots.

In [ ]:
import random
import io
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy import symbols, sqrt, log, sin, cos, tan, cot, asin, acos, atan, acot, init_printing, latex, lambdify
from datasets import Dataset, Features, Value, Image

# ==============
# Formula latex
# ==============
def GenFx():
    init_printing(use_latex=True)
    x, y = symbols('x y')

    def genera_monomio():
        # Selezione dell'argomento base (limite concettuale x, y, -x, -y)
        arg = random.choice([x, y, -x, -y])

        tipologie = [
            'sin', 'cos', 'tan', 'cot',
            'arcsin', 'arccos', 'arctan', 'arccot',
            'log', 'pow', 'sqrt'
        ]
        scelta = random.choice(tipologie)

        if scelta == 'sin': return sin(arg)
        if scelta == 'cos': return cos(arg)
        if scelta == 'tan': return tan(arg)
        if scelta == 'cot': return cot(arg)
        if scelta == 'arcsin': return asin(arg)
        if scelta == 'arccos': return acos(arg)
        if scelta == 'arctan': return atan(arg)
        if scelta == 'arccot': return acot(arg)
        if scelta == 'log':
            base = random.choice(['e', 10])
            return log(arg) if base == 'e' else log(arg, 10)
        if scelta == 'pow':
            n = random.randint(2, 5)
            return arg**n
        if scelta == 'sqrt':
            return sqrt(arg)
        return arg

    # Crea una funzione composta da 1 a 2 elementi elementari
    n_mnm = random.randint(1, 2)
    monomi = [genera_monomio() for _ in range(n_mnm)]
    f_custom = monomi[0]

    for i in range(1, n_mnm):
        op_scelto = random.choice(['+', '-', '*'])
        if op_scelto == '+': f_custom += monomi[i]
        elif op_scelto == '-': f_custom -= monomi[i]
        elif op_scelto == '*': f_custom *= monomi[i]

    # Sostituiamo y con x per permettere il plotting se presente
    f_custom = f_custom.subs(y, x)
    return f_custom

# =============
# Generatore di grafici
# =============
def GenGraph(f_input):
    x_sym = symbols('x')
    try:
        f_expr = f_input
        f_num = lambdify(x_sym, f_expr, modules=['numpy'])

        x_vals = np.linspace(0.1, 10, 1000)
        y_vals = f_num(x_vals)

        y_vals = np.array(y_vals, dtype=np.complex128)
        y_vals = np.real(y_vals)
        y_vals[~np.isfinite(y_vals)] = np.nan

        if np.all(np.isnan(y_vals)):
            return None
        else:
            plt.figure(figsize=(10, 6))
            plt.plot(x_vals, y_vals)
            plt.axhline(0, color='black', linewidth=0.5)
            plt.axvline(0, color='black', linewidth=0.5)
            plt.grid(True, linestyle='--', alpha=0.7)

            buf = io.BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)
            plt.show()
            plt.close()
            return buf.getvalue()
    except Exception as e:
        return None

# =============
# Ciclo di generazione
# =============
TIMER = 550  # Tempo di esecuzione
start_time = time.time()
data_list = []

print("Inizio generazione funzioni elementari...")
while time.time() - start_time < TIMER:
    f_expr = GenFx()
    latex_str = latex(f_expr)
    img_bytes = GenGraph(f_expr)

    if img_bytes is not None:
        data_list.append({"graph": img_bytes, "latex_formula": latex_str})

if data_list:
    df = pd.DataFrame(data_list)
    dataset = Dataset.from_pandas(df, features=Features({"graph": Image(), "latex_formula": Value("string")}))
    dataset.to_parquet("Graph-function_elementary.parquet")
    print(f"Completato! Generate {len(dataset)} funzioni.")
else:
    print("Nessun dato generato.")